In [7]:
# Example: IMDB dataset
df = pd.read_csv("Train.csv")
df_small = df.sample(10000, random_state=42)
df_small.to_csv("Train.csv", index=False)
df_small.head()

,text,label
32823,The central theme in this movie seems to be co...,0
16298,"An excellent example of ""cowboy noir"", as it's...",1
28505,The ending made my heart jump up into my throa...,0
6689,Only the chosen ones will appreciate the quali...,1
26893,"This is a really funny film, especially the se...",1


In [11]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Load dataset
df = pd.read_csv("Train.csv")
print(df.columns)   # should show ['text', 'label']

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')   # <-- this fixes your error

# Setup preprocessing
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    text = re.sub(r'[^a-zA-Z]', ' ', str(text))   # remove non-letters
    tokens = nltk.word_tokenize(text.lower())
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]
    return " ".join(tokens)

# Apply preprocessing
df['cleaned'] = df['text'].apply(preprocess)
print("Preprocessing complete")

# TF-IDF feature extraction
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['cleaned']).toarray()
y = df['label']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print(classification_report(y_test, y_pred))

# Prediction function
def predict_sentiment(text):
    cleaned = preprocess(text)
    vector = tfidf.transform([cleaned]).toarray()
    prediction = model.predict(vector)[0]
    return "Positive" if prediction == 1 else "Negative"

print(predict_sentiment("I love this product!"))
print(predict_sentiment("This movie was terrible."))


Index(['text', 'label'], dtype='object')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


Preprocessing complete
Accuracy: 0.863
F1 Score: 0.862923275135726
              precision    recall  f1-score   support

           0       0.87      0.85      0.86       970
           1       0.86      0.88      0.87      1030

    accuracy                           0.86      2000
   macro avg       0.86      0.86      0.86      2000
weighted avg       0.86      0.86      0.86      2000

Positive
Negative
